# 📅 2026-03-23 (월) 개발 노트 : 풀스택 기본 뼈대 구축 및 연동 디버깅

## 🎯 오늘의 목표
- [x] 프론트엔드(Next.js) 환경 복구 및 Tailwind CSS 디자인 시스템 연동
- [x] 백엔드(FastAPI) 추천 API 로직 구축 및 통신 허용(CORS) 설정
- [x] 프론트엔드와 백엔드 통신 연결 및 검색 결과(게임 데이터) 화면 렌더링 성공

## 🛠 진행 상황 및 핵심 기록
- `package.json` 수동 생성 및 `npm install`을 통한 프론트엔드 엔진/의존성 패키지 전면 복구 완료.
- `next.config.mjs` 확장자 변경 및 `layout.tsx` 폰트 충돌 에러(Geist -> Inter) 해결.
- Tailwind CSS 3총사(tailwindcss, postcss, autoprefixer) 설치 및 `tailwind.config.js`, `globals.css` 셋업으로 UI 디자인 완벽 적용.
- **백엔드 API 엔드포인트:** `GET http://localhost:8000/recommend?query={검색어}` 
  - 검색어(대소문자 무시)를 기반으로 게임 이름을 필터링하고 임시 `final_score`를 계산하는 로직 적용.
- **프론트엔드-백엔드 연동:** 프론트엔드의 `fetch` 함수를 통해 백엔드 데이터를 JSON으로 받아와 `results` 상태값을 업데이트하고 카드 형태로 출력함.

## 🚨 트러블슈팅 (문제 및 해결)
- **문제 1: 프론트엔드 서버 실행 시 화면에 디자인(Tailwind)이 적용되지 않고 깨지는 현상**
  - **원인:** CSS 엔진에 스타일을 맵핑할 `tailwind.config.js` 설정 파일이 누락되어 브라우저가 클래스명을 인식하지 못함.
  - **해결:** `npx tailwindcss init -p` 명령어로 설정 파일을 생성하고, `globals.css` 상단에 `@tailwind` 디렉티브 3줄을 추가하여 엔진을 연결함.

- **문제 2: 검색 버튼 클릭 시 무반응 및 백엔드에 신호가 가지 않음**
  - **원인:** 프론트엔드에서 요청할 `/recommend` API 창구가 백엔드에 없었으며, 포트가 다른 두 서버 간의 통신을 브라우저가 보안(CORS) 이유로 차단함.
  - **해결:** `main.py`에 `@app.get("/recommend")` 라우터를 추가하고, `CORSMiddleware`를 도입해 `allow_origins=["*"]`로 통신 문을 개방함. (주소도 `127.0.0.1`에서 `localhost`로 통일)

- **문제 3: 영어로 정상 검색 시 백엔드에서 500 Internal Server Error (`KeyError: 'added'`) 발생**
  - **원인:** 백엔드 점수 계산 로직에서 `added`, `rating` 데이터를 찾았으나, 실제 로드한 CSV 파일 내부에 해당 컬럼이 누락되어 있어 파이썬 판다스 엔진이 뻗어버림.
  - **해결:** 데이터 존재 여부를 먼저 확인하고, 값이 없을 경우 `.fillna(0)`을 통해 0점 처리하도록 백엔드 방탄(안전장치) 코드를 추가하여 런타임 에러를 원천 차단함.

## 💡 인사이트 및 다음 할 일
- **배운 점:** 프론트엔드와 백엔드를 연결할 때는 **CORS 설정**이 숨 쉬듯 자연스럽게 들어가야 함을 체감함. 또한, 외부 데이터(CSV)를 다룰 때는 언제든 빈 값이 들어오거나 항목이 통째로 빠져있을 수 있으므로, 서버가 멈추지 않도록 예외 처리(에러 방어 로직)를 하는 것이 백엔드 안정성의 핵심임을 깨달음.
- **다음 할 일:** 1. API 고도화: 게임 카드 UI에 RAWG API를 연결하여 진짜 게임 포스터 이미지 띄우기.
  2. 추천 로직 진화: 단순 텍스트 포함 여부가 아닌, 자연어 처리가 가능한 진짜 'AI 벡터 검색(Vector Search)' 도입 설계하기.